<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Taller Hands-On: El Guardia Dormilón, el Álbum Roto y el Detective 🕵️
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Taller Práctico Evaluativo · Módulo 08
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/Para%20Dummies/08_Temas_Avanzados_Hands_On_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a practicar? 🎈

Esta es la versión **guiada** 🧸 del [taller estándar del Módulo 08](../08_Temas_Avanzados_Hands_On.ipynb). Practicamos tres ideas de los cuadernos Para Dummies del módulo, cada una con una analogía:

1. 😴 **El guardia dormilón** ([desbalanceados](../../08%20-%20Temas%20Avanzados%20de%20ML/Para%20Dummies/03_Desbalanceados_Dummies.ipynb)): si casi nunca pasa nada, un guardia que dice «todo bien» acierta casi siempre... y no sirve. Le enseñaremos a despertar.
2. 📖 **El álbum con fotos rotas** ([datos faltantes](../../08%20-%20Temas%20Avanzados%20de%20ML/Para%20Dummies/04_Datos_Faltantes_Dummies.ipynb)): ¿rellenamos con el promedio o preguntamos a los vecinos? Y siempre **dentro** de la tubería, para no hacer trampa.
3. 🕵️ **El detective** ([explicabilidad](../../08%20-%20Temas%20Avanzados%20de%20ML/Para%20Dummies/02_Explicabilidad_Dummies.ipynb)): le tapamos una pista al modelo y vemos cuánto le hace falta.

**Cómo trabajar:** completa los huecos `...` y los `# TODO` (siempre hay una pista 💡), luego ejecuta la celda de **autoverificación**: si algo falta, te dirá qué revisar. Al final, *Restart & Run All*. No necesitas descargar nada.

---
### 📦 Los datos (ejecuta sin modificar)

Tres tablas **ficticias**: `X1`/`y1` (2 000 transacciones, ≈ 8 % fraudes), `X2`/`y2` (500 personas con casillas vacías) y `X3`/`y3` (800 máquinas, `y3 = 1` si falló).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import recall_score, roc_auc_score
import warnings; warnings.filterwarnings("ignore")
sig = lambda t: 1 / (1 + np.exp(-t))

rng = np.random.default_rng(31)
X1 = pd.DataFrame(rng.normal(size=(2000, 4)), columns=["monto", "hora", "distancia", "n_tx"])
y1 = pd.Series((rng.random(2000) < sig(1.4 * X1.monto + 1.0 * X1.distancia + 0.8 * X1.n_tx - 3.6)).astype(int), name="fraude")

rng = np.random.default_rng(32)
X2 = pd.DataFrame(rng.normal(size=(500, 4)), columns=["ingreso", "gasto", "edad", "ahorro"])
y2 = pd.Series((rng.random(500) < sig(1.2 * X2.ingreso + 1.0 * X2.gasto - 0.5 * X2.edad)).astype(int), name="compra")
for col, p in (("gasto", 0.25), ("edad", 0.15), ("ahorro", 0.20)):
    X2.loc[rng.random(500) < p, col] = np.nan

rng = np.random.default_rng(33)
X3 = pd.DataFrame(rng.normal(size=(800, 4)), columns=["temperatura", "vibracion", "ruido_a", "ruido_b"])
y3 = pd.Series((rng.random(800) < sig(2.0 * X3.temperatura + 0.9 * X3.vibracion)).astype(int), name="falla")
print("Fraudes:", round(y1.mean(), 3), "| casillas vacías en X2:", int(X2.isna().sum().sum()), "| fallas:", round(y3.mean(), 2))

---
## Reto 1: Despertar al Guardia Dormilón 😴 *(25 puntos)*

**Analogía.** Un guardia que siempre dice «todo tranquilo» acierta el 92 % de las noches (casi nunca hay ladrones), pero **nunca atrapa a ninguno**. Por eso miramos el ***recall*** (de todos los ladrones, ¿a cuántos atrapamos?) y no solo la *accuracy*.

**Qué hacer.**
1. Separa `X1`, `y1`: 30 % de prueba, `stratify=y1`, `random_state=0`.
2. `acc_dummy`: *accuracy* en prueba del guardia dormilón (`DummyClassifier(strategy="most_frequent")`).
3. Entrena la regresión logística normal (`modelo_plano`) y guarda `recall_plano` en prueba.
4. Entrena otra con `class_weight="balanced"` («le gritamos al guardia: ¡los ladrones pesan más!») y guarda `recall_pesos`.
5. Con `modelo_plano`, cambia la regla de decisión: marca fraude si la probabilidad es **≥ 0.15** (en vez de 0.5) y guarda `recall_umbral`.

**Meta.** El guardia dormilón saca más de 0.9 de *accuracy* pero ¡0 de recall!; las otras dos ideas deben atrapar claramente más fraudes que el modelo plano.

In [ ]:
# 1) Separar
X1_tr, X1_te, y1_tr, y1_te = train_test_split(..., ..., test_size=..., stratify=..., random_state=0)   # 💡 completa los huecos

# 2) El guardia dormilón
dormilon = DummyClassifier(strategy="most_frequent").fit(X1_tr, y1_tr)
acc_dummy = dormilon.score(..., ...)    # 💡 evalúalo con la prueba

# 3) Modelo plano
modelo_plano = make_pipeline(StandardScaler(), LogisticRegression()).fit(X1_tr, y1_tr)
recall_plano = recall_score(y1_te, modelo_plano.predict(X1_te))

# 4) Modelo con pesos: los fraudes "pesan más"
modelo_pesos = make_pipeline(StandardScaler(), LogisticRegression(class_weight=...)).fit(X1_tr, y1_tr)   # 💡 "balanced"
recall_pesos = None  # TODO: igual que recall_plano pero con modelo_pesos

# 5) Cambiar la regla de decisión: fraude si probabilidad >= 0.15
proba = modelo_plano.predict_proba(X1_te)[:, 1]
recall_umbral = None  # TODO: recall_score(y1_te, (proba >= 0.15).astype(int))

print("Accuracy del dormilón:", acc_dummy, "| recall plano:", recall_plano, "| con pesos:", recall_pesos, "| con umbral 0.15:", recall_umbral)

In [ ]:
# ✅ Autoverificación del Reto 1 (no modifiques esta celda)
_falta = [v for v in ("X1_tr", "X1_te", "y1_tr", "y1_te", "acc_dummy", "recall_plano", "recall_pesos", "recall_umbral",) if v not in globals() or globals()[v] is None or globals()[v] is Ellipsis]
assert not _falta, f"💡 Reto 1: todavía faltan por crear/completar: {_falta}. Reemplaza los huecos '...' y los TODO."

assert X1_tr.shape == (1400, 4) and X1_te.shape == (600, 4), "💡 Reto 1: la partición debe dejar 1400 filas para entrenar y 600 para la prueba (test_size=0.30)."
assert abs(y1_tr.mean() - y1_te.mean()) < 0.01, "💡 Reto 1: usa stratify=y1 para que la proporción de fraudes sea parecida en ambos lados."
assert acc_dummy > 0.88, "💡 Reto 1: el guardia dormilón debería tener accuracy > 0.88 (casi nunca hay fraude). Evalúa 'dormilon' con X1_te, y1_te."
assert recall_score(y1_te, dormilon.predict(X1_te)) == 0, "💡 Reto 1: el guardia dormilón atrapa 0 fraudes: ¡por eso accuracy engaña!"
assert recall_pesos > recall_plano + 0.15, "💡 Reto 1: con class_weight='balanced' el recall debe subir claramente respecto del modelo plano."
assert recall_umbral > recall_plano + 0.10, "💡 Reto 1: bajar el umbral a 0.15 debe atrapar más fraudes que el umbral 0.5. Usa (proba >= 0.15)."
print("✅ Reto 1 superado: el guardia ya no se duerme.")

---
## Reto 2: Rellenar el Álbum, pero sin Hacer Trampa 📖 *(25 puntos)*

**Analogía.** En un álbum faltan fotos. Puedes poner **la foto promedio** o **preguntarle a los vecinos más parecidos** qué había ahí. Y la regla de oro: el relleno se **aprende solo con el entrenamiento** de cada ronda de validación, así que debe vivir **dentro de la tubería** (`Pipeline`).

**Qué hacer** (con `X2`, `y2`):
1. Cuenta cuántas casillas vacías hay por columna (`vacias`) y qué fracción de las filas tiene alguna vacía (`frac_filas`).
2. Arma dos tuberías de tres pasos (imputador → `StandardScaler` → `LogisticRegression`): `pipe_media` con `SimpleImputer(strategy="mean")` y `pipe_knn` con `KNNImputer(n_neighbors=5)`.
3. Calcula el AUC medio de cada una con `cross_val_score(..., cv=cv, scoring="roc_auc")` y guárdalo en `auc_media` y `auc_knn`.

**Meta.** Las dos tuberías tienen al imputador como **primer paso**, y ambos AUC son claramente mejores que adivinar (> 0.7).

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=0)

# 1) ¿Cuántas casillas vacías?
vacias = X2.isna().sum()            # 💡 ya está: cuenta por columna
frac_filas = None  # TODO: fracción de filas con AL MENOS una casilla vacía (pista: X2.isna().any(axis=1).mean())

# 2) Dos tuberías: el imputador va PRIMERO
pipe_media = Pipeline([("imputador", SimpleImputer(strategy=...)),   # 💡 "mean"
                       ("escalador", StandardScaler()),
                       ("clf", LogisticRegression())])
pipe_knn = Pipeline([("imputador", KNNImputer(n_neighbors=...)),     # 💡 5 vecinos
                     ("escalador", StandardScaler()),
                     ("clf", LogisticRegression())])

# 3) AUC con validación cruzada
auc_media = cross_val_score(pipe_media, ..., ..., cv=cv, scoring="roc_auc").mean()   # 💡 datos y etiquetas completos
auc_knn = None  # TODO: lo mismo con pipe_knn
print(vacias.to_dict(), "| filas con alguna vacía:", frac_filas)
print("AUC media:", auc_media, "| AUC vecinos:", auc_knn)

In [ ]:
# ✅ Autoverificación del Reto 2 (no modifiques esta celda)
_falta = [v for v in ("frac_filas", "pipe_media", "pipe_knn", "auc_media", "auc_knn",) if v not in globals() or globals()[v] is None or globals()[v] is Ellipsis]
assert not _falta, f"💡 Reto 2: todavía faltan por crear/completar: {_falta}. Reemplaza los huecos '...' y los TODO."

assert vacias["ingreso"] == 0 and vacias["gasto"] > 50, "💡 Reto 2: 'vacias' debe contar las casillas vacías por columna (X2.isna().sum())."
assert 0.3 < frac_filas < 0.9, "💡 Reto 2: frac_filas debe ser la fracción de filas con AL MENOS una casilla vacía: X2.isna().any(axis=1).mean()."
assert isinstance(pipe_media, Pipeline) and isinstance(pipe_media.steps[0][1], SimpleImputer) and pipe_media.steps[0][1].strategy == "mean", "💡 Reto 2: pipe_media debe empezar con SimpleImputer(strategy='mean')."
assert isinstance(pipe_knn, Pipeline) and isinstance(pipe_knn.steps[0][1], KNNImputer) and pipe_knn.steps[0][1].n_neighbors == 5, "💡 Reto 2: pipe_knn debe empezar con KNNImputer(n_neighbors=5)."
assert 0.7 < auc_media <= 1 and 0.7 < auc_knn <= 1, "💡 Reto 2: ambos AUC deben estar entre 0.7 y 1; calcúlalos con cross_val_score sobre X2, y2 (scoring='roc_auc')."
print("✅ Reto 2 superado: rellenaste el álbum sin hacer trampa.")

---
## Reto 3: El Detective que Tapa una Pista 🕵️ *(25 puntos)*

**Analogía.** Para saber qué pista usa más un detective, se la tapas y ves cuánto empeora. En ML: **barajamos** los valores de una columna (la «tapamos») y medimos cuánto **cae el AUC**. Si no cae, el modelo ni la miraba.

**Qué hacer** (con `X3`, `y3`; el modelo ya viene entrenado):
1. `auc_original`: AUC de `modelo` sobre `X3_te`.
2. Para **cada columna**: haz una copia de `X3_te`, **baraja** esa columna con `rng.permutation(...)`, calcula el AUC con la columna barajada y guarda la **caída** `auc_original - auc_barajada` en el diccionario `caida`.
3. `mas_importante`: la columna con mayor caída.

**Meta.** `temperatura` debe ser la pista clave, y `ruido_a`/`ruido_b` casi no deben importar (caída menor que 0.03).

> ⚠️ Recuerda: que el modelo **use** una pista no significa que esa pista **cause** la falla.

In [ ]:
X3_tr, X3_te, y3_tr, y3_te = train_test_split(X3, y3, test_size=0.3, random_state=0)
modelo = make_pipeline(StandardScaler(), LogisticRegression()).fit(X3_tr, y3_tr)
rng = np.random.default_rng(0)

auc_original = roc_auc_score(y3_te, modelo.predict_proba(...)[:, 1])    # 💡 con la prueba

caida = {}
for col in X3_te.columns:
    X_tapada = X3_te.copy()                                    # copia: no dañes la original
    X_tapada[col] = rng.permutation(X_tapada[col].to_numpy())   # 💡 "tapamos" la pista barajándola
    auc_barajada = roc_auc_score(y3_te, modelo.predict_proba(...)[:, 1])   # 💡 con la copia tapada
    caida[col] = None  # TODO: auc_original - auc_barajada

mas_importante = None  # TODO: la columna con mayor caída (pista: max(caida, key=caida.get))
print(caida, "| más importante:", mas_importante)

In [ ]:
# ✅ Autoverificación del Reto 3 (no modifiques esta celda)
_falta = [v for v in ("auc_original", "caida", "mas_importante",) if v not in globals() or globals()[v] is None or globals()[v] is Ellipsis]
assert not _falta, f"💡 Reto 3: todavía faltan por crear/completar: {_falta}. Reemplaza los huecos '...' y los TODO."

assert auc_original > 0.8, "💡 Reto 3: el AUC original del modelo debería ser > 0.8; calcúlalo con modelo.predict_proba(X3_te)."
assert set(caida) == set(X3_te.columns) and all(isinstance(v, (float, np.floating)) for v in caida.values()), "💡 Reto 3: 'caida' debe tener una caída numérica para cada columna: auc_original - auc_barajada."
assert caida["temperatura"] > 0.15, "💡 Reto 3: al tapar 'temperatura' el AUC debe caer mucho (> 0.15). ¿Evaluaste con X_tapada (la copia barajada)?"
assert abs(caida["ruido_a"]) < 0.03 and abs(caida["ruido_b"]) < 0.03, "💡 Reto 3: las columnas de ruido casi no deberían afectar al AUC (caída < 0.03)."
assert caida["temperatura"] > caida["vibracion"] > 0, "💡 Reto 3: 'vibracion' aporta algo, pero menos que 'temperatura'."
assert mas_importante == "temperatura", "💡 Reto 3: mas_importante es la columna con mayor caída: max(caida, key=caida.get)."
print("✅ Reto 3 superado: ya sabes qué pista usa el modelo.")

---
## Reto 4: Mini-Reflexión ✍️ *(25 puntos)*

Escribe **3 o 4 líneas** con tus palabras (sin código): ¿por qué el guardia dormilón tiene tan buena *accuracy* y tan mal *recall*? ¿Por qué el relleno de las casillas vacías debe hacerse dentro de la tubería? Si `temperatura` es la pista más importante, ¿puedes afirmar que **causa** la falla? 🧸

#### ✍️ Tu respuesta

_Escribe aquí tus 3 o 4 líneas. Reemplaza este texto por tu respuesta._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: guardia dormilón, pesos y umbral (recall) | 25 |
| Reto 2: imputación dentro del `Pipeline` (media vs. vecinos) | 25 |
| Reto 3: importancia por permutación a mano | 25 |
| Reto 4: mini-reflexión con tus palabras | 25 |

### ✅ Checklist de entrega

- [ ] No quedan `...` ni `# TODO` sin completar.
- [ ] Las tres autoverificaciones imprimen «✅ Reto N superado».
- [ ] Hiciste *Restart & Run All* y todo corrió sin errores.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>